LangChain RAG Tool — Rewritten for LangChain v1 (langchain==1.3.11)
This notebook keeps the exact same task as the original 9_LangchainRAGTool.ipynb: build a FAISS-backed retriever over sample.txt, wrap it as a tool the agent can call, give the agent conversational memory, and ask it the same 3 questions — "What is LangChain?", "Who created it?", "Explain LangChain's use in AI workflows."

Only how it's built has changed — the legacy initialize_agent + AgentType.CONVERSATIONAL_REACT_DESCRIPTION + RetrievalQA + ConversationBufferMemory stack is replaced with create_agent:

Legacy piece	v1 replacement
RetrievalQA.from_chain_type(llm, retriever) wrapped in a Tool	A plain @tool function that calls retriever.invoke(query) and has the LLM answer from that context
ConversationBufferMemory(memory_key="chat_history", ...)	checkpointer=InMemorySaver() + thread_id
initialize_agent(..., agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, handle_parsing_errors=True)	create_agent(model, tools, checkpointer=...)
handle_parsing_errors=True (needed because the old text-based ReAct agent could fail to parse the LLM's free-form output — see the "Could not parse LLM output" error in the original notebook's own Q3 run)	Not needed — native tool calling has no free-text format to fail on
langchain.chat_models.ChatOpenAI	langchain_openai.ChatOpenAI
langchain.vectorstores.FAISS / langchain.embeddings.OpenAIEmbeddings	langchain_community.vectorstores.FAISS / langchain_openai.OpenAIEmbeddings
langchain.text_splitter.CharacterTextSplitter	langchain_text_splitters.CharacterTextSplitter


Reference: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent

In [1]:
# 📦 Install if not already done
#!pip install "langchain==1.3.11" langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken python-dotenv langgraph

# 1️⃣ Imports
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["OPEN_API_KEY"]="Open_api_key"

# 3️⃣ Setup LLM
# NOTE: the original notebook relied on the default model (gpt-3.5-turbo era),
# which has since been retired — pin an explicit, currently-supported model.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 4️⃣ Create Vector DB (Retriever)
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 5️⃣ Wrap the retriever as an agent tool (replaces RetrievalQA + Tool)
@tool
def LangChainRetriever(query: str) -> str:
    """Use this to answer questions about LangChain framework, features, or its creator."""
    docs = retriever.invoke(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(
        f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
    )
    return answer.content

# 6️⃣ Setup Memory (checkpointer, replaces ConversationBufferMemory)
checkpointer = InMemorySaver()
thread_config = {"configurable": {"thread_id": "rag-demo-1"}}

# 7️⃣ Initialize Agent with Tool + Memory (replaces initialize_agent)
agent = create_agent(
    model=llm,
    tools=[LangChainRetriever],
    checkpointer=checkpointer,
)

# 8️⃣ Ask Questions (RAG-Style)
print("1️⃣ First Question")
res1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]}, thread_config)
print("Answer:", res1["messages"][-1].content)

print("\n2️⃣ Follow-up")
res2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]}, thread_config)
print("Answer:", res2["messages"][-1].content)

print("\n3️⃣ Combined Reasoning")
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain LangChain's use in AI workflows."}]}, thread_config
)
print("Answer:", res3["messages"][-1].content)

C:\Users\KAVITHA\AppData\Local\Temp\ipykernel_20040\994654885.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1️⃣ First Question
Answer: LangChain is a framework for building applications with large language models (LLMs). It provides tools and components to facilitate the development of applications that leverage the capabilities of LLMs. If you want to know more details or specific features of LangChain, feel free to ask!

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase. If you have any more questions about LangChain or its creator, feel free to ask!

3️⃣ Combined Reasoning
Answer: LangChain is used in AI workflows by providing a framework that supports key components such as Retrieval-Augmented Generation (RAG), agents, memory, and tools. This enables developers to build applications that effectively integrate large language models (LLMs) within various AI processes. By leveraging LangChain, AI workflows can incorporate advanced language understanding, retrieval of relevant information, and dynamic interaction capabilities, making the applications more powerful and context-awa